# AdventureWorks — Exercise 1: Core ETL Pipeline

**Course:** Data Engineering  
**Notebook:** 01 — Building the Core Data Pipeline  
**Assistant:** Antigravity  

---

## What This Notebook Covers

| # | Section | Purpose |
|---|---------|----------|
| 1 | Configuration & Setup | Verify paths and import libraries |
| 2 | Source File Check | Confirm all CSV files exist |
| 3 | Data Extraction | Load all 4 tables with Pandas |
| 4 | Schema Display | Inspect column names, types, shapes |
| 5 | Record Counts | How many rows per table? |
| 6 | Sample Records | Preview actual data |
| 7 | Raw Staging | Persist raw data to staging/raw/ |

> **Architecture Note:** This notebook uses a **Full Refresh** strategy.  
> Every time you run it, the data is read fresh from the source CSVs.  
> No old data is carried forward.


---
## Section 1 — Configuration & Setup

**Objective:**  
Import required libraries and load the project configuration.  
The configuration file (`src/config.py`) is the **single source of truth** for all paths.

> **Why a config file?**  
> Instead of hard-coding `C:\Users\Kevin\Data\...` everywhere, we define `DATA_PATH` once.  
> If you move your CSV files, you only need to change one line in `config.py`.


In [1]:
# ── Standard Library ──────────────────────────────────────────
import sys
import os
from pathlib import Path
from datetime import datetime

# ── Third-Party ───────────────────────────────────────────────
import pandas as pd
import numpy as np

# ── Notebook Display Settings ─────────────────────────────────
pd.set_option('display.max_columns', 30)
pd.set_option('display.max_colwidth', 40)
pd.set_option('display.width', 120)

# ── Add project /src to Python path ──────────────────────────
NOTEBOOK_DIR = Path().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent          # AdventureWorks_DataEngineering/
SRC_DIR      = PROJECT_ROOT / 'src'
sys.path.insert(0, str(SRC_DIR))

# ── Import project config ─────────────────────────────────────
from config import (
    DATA_PATH, SOURCE_FILES,
    CSV_SEPARATOR,
    STAGING_RAW,
    PRODUCT_COLUMNS, CUSTOMER_COLUMNS,
    SALESORDERHEADER_COLUMNS, SALESORDERDETAIL_COLUMNS,
    ensure_directories,
)

print('✓ Libraries imported')
print(f'✓ Project Root : {PROJECT_ROOT}')
print(f'✓ Data Path    : {DATA_PATH}')
print(f'✓ Staging Raw  : {STAGING_RAW}')

✓ Libraries imported
✓ Project Root : F:\DE_CAT_1\AdventureWorks_DataEngineering
✓ Data Path    : F:\DE_CAT_1\AdventureWorks_DataEngineering\data\AdventureWorks
✓ Staging Raw  : F:\DE_CAT_1\AdventureWorks_DataEngineering\staging\raw


**Explanation:**  
- We add the `/src` folder to Python's module search path so `import config` works from the notebook.  
- `PROJECT_ROOT` is computed dynamically — this notebook works on any machine without path changes.  
- `DATA_PATH` is imported from `config.py`. Change it there, not here.


---
## Section 2 — Source File Check

**Objective:**  
Before reading any data, verify that all four required CSV files exist at the configured path.  
This prevents cryptic errors later in the pipeline.


In [2]:
print('=' * 65)
print('  SOURCE FILE VERIFICATION')
print('=' * 65)
print(f'  Configured DATA_PATH: {DATA_PATH}\n')

all_found = True
file_info = []

for name, path in SOURCE_FILES.items():
    exists   = path.exists()
    size_mb  = round(path.stat().st_size / 1_048_576, 2) if exists else 0
    status   = '✓ FOUND' if exists else '✗ MISSING'
    color    = '' if exists else '[!] '
    all_found = all_found and exists
    file_info.append({'Table': name, 'File': path.name, 'Status': status, 'Size (MB)': size_mb})
    print(f'  {status}  {name:20s}  {path.name:30s}  {size_mb:>6} MB')

print()
if all_found:
    print('  ✅ All source files found. Ready to extract.')
else:
    print('  ❌ One or more files are MISSING.')
    print('  → Open src/config.py and update DATA_PATH to the correct folder.')
print('=' * 65)

  SOURCE FILE VERIFICATION
  Configured DATA_PATH: F:\DE_CAT_1\AdventureWorks_DataEngineering\data\AdventureWorks

  ✓ FOUND  product               Product.csv                       0.08 MB
  ✓ FOUND  customer              Customer.csv                      1.64 MB
  ✓ FOUND  salesorderheader      SalesOrderHeader.csv              7.53 MB
  ✓ FOUND  salesorderdetail      SalesOrderDetail.csv             13.09 MB

  ✅ All source files found. Ready to extract.


**Explanation:**  
- We iterate over `SOURCE_FILES` (defined in `config.py`) which maps table names to file paths.  
- `path.exists()` checks whether the file is present on disk.  
- `path.stat().st_size` gives the file size in bytes; we convert to MB for readability.  
- If any file is missing, the pipeline should stop here rather than produce partial results.


---
## Section 3 — Data Extraction

**Objective:**  
Read all four AdventureWorks CSV files into Pandas DataFrames.

**Key facts about the source files:**  
- Separator: **TAB** (`\t`) — not comma-separated  
- No header row — the first row is already data  
- We assign column names manually using the lists in `config.py`


In [3]:
def read_source_csv(name: str, columns: list) -> pd.DataFrame:
    """
    Read a single AdventureWorks CSV (tab-separated, no header).
    
    Parameters
    ----------
    name    : key in SOURCE_FILES dict, e.g. 'product'
    columns : list of column names to assign positionally
    
    Returns
    -------
    pd.DataFrame
    """
    path = SOURCE_FILES[name]
    
    df = pd.read_csv(
        path,
        sep=CSV_SEPARATOR,    # Tab separator
        header=None,          # No header row in file
        names=columns,        # Assign column names
        low_memory=False,
        encoding='utf-8',
        on_bad_lines='warn',  # Log bad lines, don't crash
    )
    
    # Add metadata columns for lineage
    df['_source_file'] = path.name
    df['_ingested_at'] = datetime.utcnow().isoformat(timespec='seconds')
    
    return df

print('✓ read_source_csv() helper defined')

✓ read_source_csv() helper defined


In [4]:
print('Extracting all source tables…\n')

start_time = datetime.utcnow()

# ── Extract each table ────────────────────────────────────────
df_product            = read_source_csv('product',          PRODUCT_COLUMNS)
df_customer           = read_source_csv('customer',         CUSTOMER_COLUMNS)
df_salesorderheader   = read_source_csv('salesorderheader', SALESORDERHEADER_COLUMNS)
df_salesorderdetail   = read_source_csv('salesorderdetail', SALESORDERDETAIL_COLUMNS)

end_time = datetime.utcnow()
elapsed  = (end_time - start_time).total_seconds()

print(f'  ✓ Product            → {len(df_product):>6,} records  |  {len(df_product.columns)} columns')
print(f'  ✓ Customer           → {len(df_customer):>6,} records  |  {len(df_customer.columns)} columns')
print(f'  ✓ SalesOrderHeader   → {len(df_salesorderheader):>6,} records  |  {len(df_salesorderheader.columns)} columns')
print(f'  ✓ SalesOrderDetail   → {len(df_salesorderdetail):>6,} records  |  {len(df_salesorderdetail.columns)} columns')
print(f'\n  Extraction complete in {elapsed:.2f} seconds.')

Extracting all source tables…



C:\Users\KEVIN\AppData\Local\Temp\ipykernel_11340\3563568161.py:3: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  start_time = datetime.utcnow()
C:\Users\KEVIN\AppData\Local\Temp\ipykernel_11340\3009655187.py:28: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  df['_ingested_at'] = datetime.utcnow().isoformat(timespec='seconds')
C:\Users\KEVIN\AppData\Local\Temp\ipykernel_11340\3009655187.py:28: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  df['_ingested_at'] = datetime.utcnow().isoformat(timespec='seconds')
C:\Users\KEVI

  ✓ Product            →    504 records  |  27 columns
  ✓ Customer           → 19,820 records  |  9 columns
  ✓ SalesOrderHeader   → 31,465 records  |  28 columns
  ✓ SalesOrderDetail   → 121,317 records  |  13 columns

  Extraction complete in 0.29 seconds.


C:\Users\KEVIN\AppData\Local\Temp\ipykernel_11340\3009655187.py:28: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  df['_ingested_at'] = datetime.utcnow().isoformat(timespec='seconds')
C:\Users\KEVIN\AppData\Local\Temp\ipykernel_11340\3563568161.py:11: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  end_time = datetime.utcnow()


**Explanation:**  
- `pd.read_csv()` with `sep='\t'` handles tab-separated files.  
- `header=None` tells Pandas the file has no column header row.  
- `names=columns` assigns column names in positional order (column 0 = first name, etc.).  
- `_source_file` and `_ingested_at` are **lineage metadata** — they record where each row came from and when it was loaded. This is a best practice in data engineering.


---
## Section 4 — Schema Display

**Objective:**  
Inspect the structure (schema) of each table: column names, data types, and null counts.

Understanding the schema is the **first step** in any data engineering project.  
You need to know what you have before you can transform it.


In [5]:
def show_schema(df: pd.DataFrame, table_name: str) -> None:
    """
    Display a formatted schema for a DataFrame.
    Shows: column index, name, data type, non-null count, null count, null%.
    """
    data_cols = [c for c in df.columns if not c.startswith('_')]   # exclude metadata
    
    print(f'\n{"=" * 70}')
    print(f'  SCHEMA: {table_name}')
    print(f'  Rows: {len(df):,}   |   Columns: {len(data_cols)}')
    print(f'  {"-" * 66}')
    print(f'  {"#":<4} {"Column Name":<35} {"Dtype":<15} {"Non-Null":>8} {"Nulls":>6} {"Null%":>6}')
    print(f'  {"-" * 66}')
    
    for i, col in enumerate(data_cols):
        non_null  = df[col].notna().sum()
        nulls     = df[col].isna().sum()
        null_pct  = round(nulls / len(df) * 100, 1) if len(df) > 0 else 0.0
        dtype_str = str(df[col].dtype)
        print(f'  {i:<4} {col:<35} {dtype_str:<15} {non_null:>8,} {nulls:>6,} {null_pct:>5.1f}%')
    
    print(f'  {"=" * 66}')

print('✓ show_schema() helper defined')

✓ show_schema() helper defined


In [6]:
# ── Product Schema ────────────────────────────────────────────
show_schema(df_product, 'Product')


  SCHEMA: Product
  Rows: 504   |   Columns: 25
  ------------------------------------------------------------------
  #    Column Name                         Dtype           Non-Null  Nulls  Null%
  ------------------------------------------------------------------
  0    ProductID                           int64                504      0   0.0%
  1    Name                                object               504      0   0.0%
  2    ProductNumber                       object               504      0   0.0%
  3    MakeFlag                            int64                504      0   0.0%
  4    FinishedGoodsFlag                   int64                504      0   0.0%
  5    Color                               object               256    248  49.2%
  6    SafetyStockLevel                    int64                504      0   0.0%
  7    ReorderPoint                        int64                504      0   0.0%
  8    StandardCost                        float64              504      0 

In [7]:
# ── Customer Schema ───────────────────────────────────────────
show_schema(df_customer, 'Customer')


  SCHEMA: Customer
  Rows: 19,820   |   Columns: 7
  ------------------------------------------------------------------
  #    Column Name                         Dtype           Non-Null  Nulls  Null%
  ------------------------------------------------------------------
  0    CustomerID                          int64             19,820      0   0.0%
  1    PersonID                            float64           19,119    701   3.5%
  2    StoreID                             float64            1,336 18,484  93.3%
  3    TerritoryID                         int64             19,820      0   0.0%
  4    AccountNumber                       object            19,820      0   0.0%
  5    rowguid                             object            19,820      0   0.0%
  6    ModifiedDate                        object            19,820      0   0.0%


In [8]:
# ── SalesOrderHeader Schema ───────────────────────────────────
show_schema(df_salesorderheader, 'SalesOrderHeader')


  SCHEMA: SalesOrderHeader
  Rows: 31,465   |   Columns: 26
  ------------------------------------------------------------------
  #    Column Name                         Dtype           Non-Null  Nulls  Null%
  ------------------------------------------------------------------
  0    SalesOrderID                        int64             31,465      0   0.0%
  1    RevisionNumber                      int64             31,465      0   0.0%
  2    OrderDate                           object            31,465      0   0.0%
  3    DueDate                             object            31,465      0   0.0%
  4    ShipDate                            object            31,465      0   0.0%
  5    Status                              int64             31,465      0   0.0%
  6    OnlineOrderFlag                     int64             31,465      0   0.0%
  7    SalesOrderNumber                    object            31,465      0   0.0%
  8    PurchaseOrderNumber                 object             3

In [9]:
# ── SalesOrderDetail Schema ───────────────────────────────────
show_schema(df_salesorderdetail, 'SalesOrderDetail')


  SCHEMA: SalesOrderDetail
  Rows: 121,317   |   Columns: 11
  ------------------------------------------------------------------
  #    Column Name                         Dtype           Non-Null  Nulls  Null%
  ------------------------------------------------------------------
  0    SalesOrderID                        int64            121,317      0   0.0%
  1    SalesOrderDetailID                  int64            121,317      0   0.0%
  2    CarrierTrackingNumber               object            60,919 60,398  49.8%
  3    OrderQty                            int64            121,317      0   0.0%
  4    ProductID                           int64            121,317      0   0.0%
  5    SpecialOfferID                      int64            121,317      0   0.0%
  6    UnitPrice                           float64          121,317      0   0.0%
  7    UnitPriceDiscount                   float64          121,317      0   0.0%
  8    LineTotal                           float64          12

**Explanation:**  
- `df.dtypes` returns a Series with column names as the index and data types as values.  
- `df[col].isna().sum()` counts missing (NaN/None) values per column.  
- At this stage, all numeric-looking columns are still `object` (string) because we haven't performed type conversion yet. That happens in the Transformation step (Exercise 1, Step 2).  
- High null percentages in columns like `Color`, `Size`, `Weight` are expected — many AdventureWorks products don't have these attributes.


---
## Section 5 — Record Counts

**Objective:**  
Summarize the size of each table at a glance.

Record counts are the simplest but most important data quality check —  
if a table suddenly has 0 rows or 10× the expected rows, something is wrong.


In [10]:
import datetime as dt

tables = {
    'Product':          df_product,
    'Customer':         df_customer,
    'SalesOrderHeader': df_salesorderheader,
    'SalesOrderDetail': df_salesorderdetail,
}

print('\n' + '=' * 55)
print('  RECORD COUNT SUMMARY')
print('=' * 55)
print(f'  {"Table":<22} {"Rows":>10}  {"Columns":>8}  {"Memory":>10}')
print('  ' + '-' * 51)

total_rows = 0
for name, df in tables.items():
    rows    = len(df)
    cols    = len(df.columns)
    mem_kb  = round(df.memory_usage(deep=True).sum() / 1024, 1)
    total_rows += rows
    print(f'  {name:<22} {rows:>10,}  {cols:>8}  {mem_kb:>8.1f} KB')

print('  ' + '-' * 51)
print(f'  {"TOTAL ROWS":<22} {total_rows:>10,}')
print('=' * 55)
print(f'  Extracted at: {datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S")} UTC')


  RECORD COUNT SUMMARY
  Table                        Rows   Columns      Memory
  ---------------------------------------------------
  Product                       504        27     444.8 KB
  Customer                   19,820         9    7297.1 KB
  SalesOrderHeader           31,465        28   26471.8 KB
  SalesOrderDetail          121,317        13   47930.1 KB
  ---------------------------------------------------
  TOTAL ROWS                173,106
  Extracted at: 2026-08-12 15:07:09 UTC


C:\Users\KEVIN\AppData\Local\Temp\ipykernel_11340\1209854021.py:27: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  print(f'  Extracted at: {datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S")} UTC')


**Explanation:**  
- `len(df)` returns the number of rows.  
- `df.memory_usage(deep=True).sum()` returns the total memory used by the DataFrame in bytes.  
- `deep=True` is important for object columns — without it, string columns are under-counted.  
- In a production pipeline, record counts are logged at each stage. A sudden drop (e.g., source file truncated) should trigger an alert.


---
## Section 6 — Sample Records

**Objective:**  
Preview the first few rows of each table to visually confirm that  
column names were assigned correctly and data looks reasonable.

> **Tip:** Always look at sample data before writing any transformation.  
> Real-world data is messy — looking at samples surfaces surprises early.


In [11]:
# Helper to display samples without the internal metadata columns
def show_sample(df: pd.DataFrame, table_name: str, n: int = 5) -> None:
    data_cols = [c for c in df.columns if not c.startswith('_')]
    print(f'\n── {table_name} — first {n} rows ──')
    display(df[data_cols].head(n))

show_sample(df_product, 'Product')


── Product — first 5 rows ──


,ProductID,Name,ProductNumber,MakeFlag,FinishedGoodsFlag,Color,SafetyStockLevel,ReorderPoint,StandardCost,ListPrice,Size,SizeUnitMeasureCode,WeightUnitMeasureCode,Weight,DaysToManufacture,ProductLine,Class,Style,ProductSubcategoryID,ProductModelID,SellStartDate,SellEndDate,DiscontinuedDate,rowguid,ModifiedDate
0,1,Adjustable Race,AR-5381,0,0,NaN,1000,750,0.0,0.0,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,2019-04-30 00:00:00.000,NaN,NaN,694215B7-08F7-4C0D-ACB1-D734BA44C0C8,2025-02-07 10:01:36.827
1,2,Bearing Ball,BA-8327,0,0,NaN,1000,750,0.0,0.0,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,2019-04-30 00:00:00.000,NaN,NaN,58AE3C20-4F3A-4749-A7D4-D568806CC537,2025-02-07 10:01:36.827
2,3,BB Ball Bearing,BE-2349,1,0,NaN,800,600,0.0,0.0,NaN,NaN,NaN,NaN,1,NaN,NaN,NaN,NaN,NaN,2019-04-30 00:00:00.000,NaN,NaN,9C21AED2-5BFA-4F18-BCB8-F11638DC2E4E,2025-02-07 10:01:36.827
3,4,Headset Ball Bearings,BE-2908,0,0,NaN,800,600,0.0,0.0,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,2019-04-30 00:00:00.000,NaN,NaN,ECFED6CB-51FF-49B5-B06C-7D8AC834DB8B,2025-02-07 10:01:36.827
4,316,Blade,BL-2036,1,0,NaN,800,600,0.0,0.0,NaN,NaN,NaN,NaN,1,NaN,NaN,NaN,NaN,NaN,2019-04-30 00:00:00.000,NaN,NaN,E73E9750-603B-4131-89F5-3DD15ED5FF80,2025-02-07 10:01:36.827


In [12]:
show_sample(df_customer, 'Customer')


── Customer — first 5 rows ──


,CustomerID,PersonID,StoreID,TerritoryID,AccountNumber,rowguid,ModifiedDate
0,1,NaN,934.0,1,AW00000001,3F5AE95E-B87D-4AED-95B4-C3797AFCB74F,2025-09-11 11:15:07.263
1,2,NaN,1028.0,1,AW00000002,E552F657-A9AF-4A7D-A645-C429D6E02491,2025-09-11 11:15:07.263
2,3,NaN,642.0,4,AW00000003,130774B1-DB21-4EF3-98C8-C104BCD6ED6D,2025-09-11 11:15:07.263
3,4,NaN,932.0,4,AW00000004,FF862851-1DAA-4044-BE7C-3E85583C054D,2025-09-11 11:15:07.263
4,5,NaN,1026.0,4,AW00000005,83905BDC-6F5E-4F71-B162-C98DA069F38A,2025-09-11 11:15:07.263


In [13]:
show_sample(df_salesorderheader, 'SalesOrderHeader')


── SalesOrderHeader — first 5 rows ──


,SalesOrderID,RevisionNumber,OrderDate,DueDate,ShipDate,Status,OnlineOrderFlag,SalesOrderNumber,PurchaseOrderNumber,AccountNumber,CustomerID,SalesPersonID,TerritoryID,BillToAddressID,ShipToAddressID,ShipMethodID,CreditCardID,CreditCardApprovalCode,CurrencyRateID,SubTotal,TaxAmt,Freight,TotalDue,Comment,rowguid,ModifiedDate
0,43659,10,2022-05-30 00:00:00.000,2022-06-11 00:00:00.000,2022-06-06 00:00:00.000,5,0,SO43659,PO522145787,10-4020-000676,29825,279.0,5,985,985,5,16281.0,105041Vi84182,NaN,20565.6206,1971.5149,616.0984,23153.2339,NaN,79B65321-39CA-4115-9CBA-8FE0903E12E6,2022-06-06 00:00:00.000
1,43660,10,2022-05-30 00:00:00.000,2022-06-11 00:00:00.000,2022-06-06 00:00:00.000,5,0,SO43660,PO18850127500,10-4020-000117,29672,279.0,5,921,921,5,5618.0,115213Vi29411,NaN,1294.2529,124.2483,38.8276,1457.3288,NaN,738DC42D-D03B-48A1-9822-F95A67EA7389,2022-06-06 00:00:00.000
2,43661,10,2022-05-30 00:00:00.000,2022-06-11 00:00:00.000,2022-06-06 00:00:00.000,5,0,SO43661,PO18473189620,10-4020-000442,29734,282.0,6,517,517,5,1346.0,85274Vi6854,4.0,32726.4786,3153.7696,985.5530,36865.8012,NaN,D91B9131-18A4-4A11-BC3A-90B6F53E9D74,2022-06-06 00:00:00.000
3,43662,10,2022-05-30 00:00:00.000,2022-06-11 00:00:00.000,2022-06-06 00:00:00.000,5,0,SO43662,PO18444174044,10-4020-000227,29994,282.0,6,482,482,5,10456.0,125295Vi53935,4.0,28832.5289,2775.1646,867.2389,32474.9324,NaN,4A1ECFC0-CC3A-4740-B028-1C50BB48711C,2022-06-06 00:00:00.000
4,43663,10,2022-05-30 00:00:00.000,2022-06-11 00:00:00.000,2022-06-06 00:00:00.000,5,0,SO43663,PO18009186470,10-4020-000510,29565,276.0,4,1073,1073,5,4322.0,45303Vi22691,NaN,419.4589,40.2681,12.5838,472.3108,NaN,9B1E7A40-6AE0-4AD3-811C-A64951857C4B,2022-06-06 00:00:00.000


In [14]:
show_sample(df_salesorderdetail, 'SalesOrderDetail')


── SalesOrderDetail — first 5 rows ──


,SalesOrderID,SalesOrderDetailID,CarrierTrackingNumber,OrderQty,ProductID,SpecialOfferID,UnitPrice,UnitPriceDiscount,LineTotal,rowguid,ModifiedDate
0,43659,1,4911-403C-98,1,776,1,2024.994,0.0,2024.994,B207C96D-D9E6-402B-8470-2CC176C42283,2022-05-30 00:00:00.000
1,43659,2,4911-403C-98,3,777,1,2024.994,0.0,6074.982,7ABB600D-1E77-41BE-9FE5-B9142CFC08FA,2022-05-30 00:00:00.000
2,43659,3,4911-403C-98,1,778,1,2024.994,0.0,2024.994,475CF8C6-49F6-486E-B0AD-AFC6A50CDD2F,2022-05-30 00:00:00.000
3,43659,4,4911-403C-98,1,771,1,2039.994,0.0,2039.994,04C4DE91-5815-45D6-8670-F462719FBCE3,2022-05-30 00:00:00.000
4,43659,5,4911-403C-98,1,772,1,2039.994,0.0,2039.994,5A74C7D2-E641-438E-A7AC-37BF23280301,2022-05-30 00:00:00.000


In [15]:
# Statistical summary of key numeric-looking columns in Product
print('\n── Product — Basic Statistics for key columns ──')
key_cols = ['ProductID', 'SafetyStockLevel', 'ReorderPoint',
            'StandardCost', 'ListPrice', 'DaysToManufacture']
# Convert to numeric first (they come in as strings)
product_stats = df_product[key_cols].apply(pd.to_numeric, errors='coerce')
display(product_stats.describe().round(2))


── Product — Basic Statistics for key columns ──


,ProductID,SafetyStockLevel,ReorderPoint,StandardCost,ListPrice,DaysToManufacture
count,504.00,504.00,504.00,504.00,504.00,504.00
mean,673.04,535.15,401.36,258.60,438.67,1.10
std,229.37,374.11,280.58,461.63,773.60,1.49
min,1.00,4.00,3.00,0.00,0.00,0.00
25%,447.75,100.00,75.00,0.00,0.00,0.00
50%,747.50,500.00,375.00,23.37,49.99,1.00
75%,873.25,1000.00,750.00,317.08,564.99,1.00
max,999.00,1000.00,750.00,2171.29,3578.27,4.00


**Explanation:**  
- `display()` renders DataFrames as nicely formatted HTML tables inside Jupyter.  
- We filter out `_source_file` and `_ingested_at` columns using `not c.startswith('_')`.  
- `pd.to_numeric(errors='coerce')` converts values to numbers, turning non-numeric values to `NaN` rather than raising an error — useful for spotting data quality issues.  
- `describe()` gives count, mean, std, min, max, and quartiles — a fast way to spot outliers and range issues.


---
## Section 7 — Raw Staging

**Objective:**  
Persist the raw extracted DataFrames to `staging/raw/` in **Parquet format**.

**Why Parquet?**  
- Faster to read than CSV (columnar storage)  
- Preserves data types  
- Much smaller file size than CSV  
- Industry standard in data engineering

**Why a raw staging layer?**  
- It preserves the exact source data before any transformation.  
- If a transformation later proves wrong, you can re-run from staging without re-reading CSVs.  
- Acts as an audit trail of what was ingested and when.

**Full Refresh behavior:**  
> Every time the pipeline runs, staging/raw/ files are **overwritten** completely.  
> They always represent the current state of the source CSVs.


In [16]:
import pyarrow   # Required by pandas for Parquet write

# Ensure staging directories exist
ensure_directories()

# Map: (DataFrame, raw table name)
raw_tables = [
    (df_product,          'raw_product'),
    (df_customer,         'raw_customer'),
    (df_salesorderheader, 'raw_salesorderheader'),
    (df_salesorderdetail, 'raw_salesorderdetail'),
]

print('Writing raw staging files…\n')
print(f'  Target directory: {STAGING_RAW}\n')

staged_paths = {}
for df, table_name in raw_tables:
    out_path = STAGING_RAW / f'{table_name}.parquet'
    
    # FULL REFRESH: always overwrite, never append
    df.to_parquet(out_path, index=False, engine='pyarrow')
    
    size_kb  = round(out_path.stat().st_size / 1024, 1)
    staged_paths[table_name] = out_path
    print(f'  ✓ {table_name:<30}  →  {out_path.name:<40} ({size_kb:>8.1f} KB)')

print('\n  Raw staging complete.')

Writing raw staging files…

  Target directory: F:\DE_CAT_1\AdventureWorks_DataEngineering\staging\raw

  ✓ raw_product                     →  raw_product.parquet                      (    52.2 KB)
  ✓ raw_customer                    →  raw_customer.parquet                     (  1118.4 KB)
  ✓ raw_salesorderheader            →  raw_salesorderheader.parquet             (  3108.1 KB)
  ✓ raw_salesorderdetail            →  raw_salesorderdetail.parquet             (  5891.1 KB)

  Raw staging complete.


In [17]:
# Verify: Read back one file and confirm row count matches
print('\n── Verification: Read-back check ──')
print('(Confirms staging files were written correctly)\n')

verify_pairs = [
    ('raw_product',          df_product,          'Product'),
    ('raw_customer',         df_customer,          'Customer'),
    ('raw_salesorderheader', df_salesorderheader,  'SalesOrderHeader'),
    ('raw_salesorderdetail', df_salesorderdetail,  'SalesOrderDetail'),
]

all_ok = True
for table_name, original_df, label in verify_pairs:
    path     = STAGING_RAW / f'{table_name}.parquet'
    reloaded = pd.read_parquet(path)
    match    = len(reloaded) == len(original_df)
    status   = '✓ MATCH' if match else '✗ MISMATCH'
    all_ok   = all_ok and match
    print(f'  {status}  {label:<22}  original={len(original_df):>6,}  staged={len(reloaded):>6,}')

print()
if all_ok:
    print('  ✅ All raw staging files verified successfully.')
else:
    print('  ❌ Verification FAILED — check file write permissions and disk space.')


── Verification: Read-back check ──
(Confirms staging files were written correctly)

  ✓ MATCH  Product                 original=   504  staged=   504
  ✓ MATCH  Customer                original=19,820  staged=19,820
  ✓ MATCH  SalesOrderHeader        original=31,465  staged=31,465
  ✓ MATCH  SalesOrderDetail        original=121,317  staged=121,317

  ✅ All raw staging files verified successfully.


**Explanation:**  
- `df.to_parquet(path, index=False)` saves the DataFrame without writing the Pandas row index.  
- `engine='pyarrow'` uses the Apache Arrow library for fast, efficient Parquet I/O.  
- The verification step reads each Parquet file back and compares row counts with the original DataFrame. This is a basic **data quality gate** — a pattern you'll use throughout the pipeline.  
- Notice how Parquet files are significantly smaller than the source CSV files while storing the same data.


---
## Section 8 — Step 1 Summary

**Objective:**  
Print a final summary of everything accomplished in this step.


In [18]:
print('\n' + '=' * 60)
print('  STEP 1 COMPLETE — CORE ETL SUMMARY')
print('=' * 60)
print()
print('  Source Path   :', DATA_PATH)
print('  Staging (Raw) :', STAGING_RAW)
print()
print('  ┌─────────────────────┬────────────┬──────────┐')
print('  │ Table               │ Records    │ Staging  │')
print('  ├─────────────────────┼────────────┼──────────┤')
print(f'  │ Product             │ {len(df_product):>10,} │ Parquet  │')
print(f'  │ Customer            │ {len(df_customer):>10,} │ Parquet  │')
print(f'  │ SalesOrderHeader    │ {len(df_salesorderheader):>10,} │ Parquet  │')
print(f'  │ SalesOrderDetail    │ {len(df_salesorderdetail):>10,} │ Parquet  │')
print('  └─────────────────────┴────────────┴──────────┘')
print()
print('  What was accomplished:')
print('    ✓ Configuration loaded from src/config.py')
print('    ✓ All 4 source CSV files verified')
print('    ✓ Data extracted with correct column names')
print('    ✓ Schemas displayed')
print('    ✓ Record counts verified')
print('    ✓ Sample records previewed')
print('    ✓ Raw staging files written to staging/raw/')
print('    ✓ Staging files verified (read-back check)')
print()
print('  Next: Step 2 — Data Transformation & Validation')
print('=' * 60)


  STEP 1 COMPLETE — CORE ETL SUMMARY

  Source Path   : F:\DE_CAT_1\AdventureWorks_DataEngineering\data\AdventureWorks
  Staging (Raw) : F:\DE_CAT_1\AdventureWorks_DataEngineering\staging\raw

  ┌─────────────────────┬────────────┬──────────┐
  │ Table               │ Records    │ Staging  │
  ├─────────────────────┼────────────┼──────────┤
  │ Product             │        504 │ Parquet  │
  │ Customer            │     19,820 │ Parquet  │
  │ SalesOrderHeader    │     31,465 │ Parquet  │
  │ SalesOrderDetail    │    121,317 │ Parquet  │
  └─────────────────────┴────────────┴──────────┘

  What was accomplished:
    ✓ Configuration loaded from src/config.py
    ✓ All 4 source CSV files verified
    ✓ Data extracted with correct column names
    ✓ Schemas displayed
    ✓ Record counts verified
    ✓ Sample records previewed
    ✓ Raw staging files written to staging/raw/
    ✓ Staging files verified (read-back check)

  Next: Step 2 — Data Transformation & Validation


---
## Concepts Learned in This Notebook

| Concept | What It Means |
|---------|---------------|
| **Data Extraction** | Reading raw data from source systems (files, databases, APIs) |
| **Configuration-Driven Design** | Using a config file to avoid hard-coded paths |
| **Tab-Separated Values (TSV)** | CSV variant using tab as delimiter instead of comma |
| **Schema** | The structure of a table: column names and data types |
| **Null / Missing Values** | Fields with no value — important to detect early |
| **Raw Staging Layer** | A copy of source data before transformation — preserves original state |
| **Parquet Format** | Columnar binary format — faster and smaller than CSV |
| **Full Refresh** | Every run starts from scratch — no dependency on previous runs |
| **Lineage Metadata** | `_source_file`, `_ingested_at` — track where data came from |
| **Verification / Data Quality Gate** | Checking outputs match expectations before proceeding |

---
*Next notebook:* `02_schema_design.ipynb` — OLTP and Star Schema design  
*Developed with:* **Antigravity** — AI Coding Assistant
